# C-ITS 스냅샷 생명주기 감사 v0.1

저장된 실행 보고서를 이용해 정상 빈 응답, 실패, 레코드 수와 생명주기 전이를 검토한다. 인증키와 원본 요청 URL은 읽지 않는다.

In [1]:
import json
from pathlib import Path

project_root = Path.cwd()
if project_root.name == '03_notebooks':
    project_root = project_root.parent
monitor_root = project_root / '02_outputs/data/cits_snapshot_monitor_v0_1'
run_paths = sorted((monitor_root / 'runs').glob('*_run_report.json'))
reports = [json.loads(path.read_text(encoding='utf-8')) for path in run_paths]
len(reports), [report['run_id'] for report in reports]


(3,
 ['20261027T093230723234+0900',
  '20261027T093233345410+0900',
  '20261027T093752984235+0900'])

In [2]:
run_summary = [
    {
        'run_id': report['run_id'],
        'collected_at': report['collected_at'],
        'status': report['status'],
        'failed_sources': report['failed_source_count'],
        'quality_failed_sources': report['quality_failed_source_count'],
        **report['transition_counts'],
    }
    for report in reports
]
run_summary


[{'run_id': '20261027T093230723234+0900',
  'collected_at': '2026-10-27T09:32:30.723234+09:00',
  'status': 'pass',
  'failed_sources': 0,
  'quality_failed_sources': 0,
  'new': 46},
 {'run_id': '20261027T093233345410+0900',
  'collected_at': '2026-10-27T09:32:33.345410+09:00',
  'status': 'pass',
  'failed_sources': 0,
  'quality_failed_sources': 0,
  'unchanged': 46},
 {'run_id': '20261027T093752984235+0900',
  'collected_at': '2026-10-27T09:37:52.984235+09:00',
  'status': 'pass',
  'failed_sources': 0,
  'quality_failed_sources': 0,
  'changed': 4,
  'disappeared': 4,
  'new': 5,
  'unchanged': 38}]

In [3]:
source_summary = []
for report in reports:
    for source in report['sources']:
        quality = source['quality']
        source_summary.append({
            'run_id': report['run_id'],
            'source_id': source['source_id'],
            'snapshot_status': source['snapshot_status'],
            'quality_status': quality['status'],
            'record_count': quality['record_count'],
            'active_records': quality['active_record_count'],
            'reported_count_matches': quality['reported_count_matches'],
            'transitions': quality['transition_counts'],
        })
source_summary


[{'run_id': '20261027T093230723234+0900',
  'source_id': 'jeju_cits_road_event',
  'snapshot_status': 'success',
  'quality_status': 'pass',
  'record_count': 33,
  'active_records': 33,
  'reported_count_matches': True,
  'transitions': {'new': 33}},
 {'run_id': '20261027T093230723234+0900',
  'source_id': 'jeju_cits_road_close',
  'snapshot_status': 'success',
  'quality_status': 'pass',
  'record_count': 0,
  'active_records': 0,
  'reported_count_matches': True,
  'transitions': {}},
 {'run_id': '20261027T093230723234+0900',
  'source_id': 'jeju_cits_road_work',
  'snapshot_status': 'success',
  'quality_status': 'pass',
  'record_count': 1,
  'active_records': 1,
  'reported_count_matches': True,
  'transitions': {'new': 1}},
 {'run_id': '20261027T093230723234+0900',
  'source_id': 'jeju_cits_road_weather',
  'snapshot_status': 'success',
  'quality_status': 'pass',
  'record_count': 12,
  'active_records': 12,
  'reported_count_matches': True,
  'transitions': {'new': 12}},
 {'ru

In [4]:
assert len(reports) >= 2
assert all(report['status'] == 'pass' for report in reports)
assert all(report['failed_source_count'] == 0 for report in reports)
assert all(report['quality_failed_source_count'] == 0 for report in reports)
assert any(
    row['source_id'] == 'jeju_cits_road_close'
    and row['record_count'] == 0
    and row['snapshot_status'] == 'success'
    for row in source_summary
)
'Snapshot quality gates passed; provider event freshness remains unverified.'


'Snapshot quality gates passed; provider event freshness remains unverified.'